# Optimizers Compare (AdamW vs MagmaAdamW)

Один и тот же датасет и модель, два оптимизатора — сравнение кривых loss с tqdm и plotly.

In [ ]:
from pathlib import Path
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm
import copy
import plotly.graph_objects as go
from transformers import AutoTokenizer, DataCollatorForLanguageModeling

from homellm.models.home_model import HomeConfig, HomeForCausalLM
from homellm.training.pretrain import StreamingTextDataset
from homellm.training.optimizers import MagmaAdamW

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DATA_PATH = '/app/datasets/fineweb-2_train.jsonl'
ensure_pretrain_dataset(DATA_PATH)  # скачает с HF, если файла нет
SEQ_LEN = 512
BATCH_SIZE = 2
STEPS = 80
LR = 3e-4

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('gpt2')
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({'pad_token': '<|pad|>'})
dataset = StreamingTextDataset(DATA_PATH, tokenizer, seq_len=SEQ_LEN)
collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, collate_fn=collator, num_workers=0)

cfg = HomeConfig(vocab_size=len(tokenizer), hidden_size=256, num_hidden_layers=4, num_attention_heads=4, max_position_embeddings=SEQ_LEN)
model_a = HomeForCausalLM(cfg).to(DEVICE)
model_m = copy.deepcopy(model_a).to(DEVICE)
model_m.resize_token_embeddings(len(tokenizer))
model_a.resize_token_embeddings(len(tokenizer))

opt_a = torch.optim.AdamW(model_a.parameters(), lr=LR, weight_decay=0.1)
opt_m = MagmaAdamW(model_m.parameters(), lr=LR, weight_decay=0.1, magma_prob=0.5)
print('Two models ready: AdamW vs MagmaAdamW')

In [ ]:
def run_steps(model, opt, name, steps=STEPS):
    model.train()
    losses = []
    it = iter(loader)
    for _ in tqdm(range(steps), desc=name):
        batch = next(it, None)
        if batch is None:
            it = iter(loader)
            batch = next(it)
        out = model(input_ids=batch['input_ids'].to(DEVICE), labels=batch['labels'].to(DEVICE))
        out.loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        opt.zero_grad(set_to_none=True)
        losses.append(out.loss.item())
    return losses

loss_a = run_steps(model_a, opt_a, 'AdamW')
loss_m = run_steps(model_m, opt_m, 'MagmaAdamW')

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(y=loss_a, mode='lines', name='AdamW'))
fig.add_trace(go.Scatter(y=loss_m, mode='lines', name='MagmaAdamW'))
fig.update_layout(title='Loss: AdamW vs MagmaAdamW', xaxis_title='step', yaxis_title='loss', template='plotly_dark', height=400)
fig.show()